# Chapter 25 — Differential Equations and Flows

Companion notebook to Chapter 25 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Flows are computed with the classical Runge–Kutta method of order 4, written
in PyTorch so that automatic differentiation goes through it. Its error is
far below the tolerances used; where two quantities are equal only up to
this error, the tolerance says so.

In [1]:
import itertools
import math
import warnings

import numpy as np
import torch
from torch.func import jacrev, vmap

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def sgn(perm):
    n_inv = sum(1 for i in range(len(perm)) for j in range(i + 1, len(perm)) if perm[i] > perm[j])
    return -1 if n_inv % 2 else 1


def antisym(t):
    k = t.dim()
    if k < 2:
        return t.clone()
    return sum(sgn(p) * t.permute(*p) for p in itertools.permutations(range(k)))


def wedge(a, b):
    return antisym(torch.tensordot(a, b, dims=0)) / (math.factorial(a.dim()) * math.factorial(b.dim()))


def dx(i, n):
    v = torch.zeros(n)
    v[i] = 1.0
    return v


def d(omega):
    def d_omega(x):
        J = jacrev(omega)(x)
        k = J.dim() - 1
        Tn = J.movedim(-1, 0)
        return sum((-1) ** j * Tn.movedim(0, j) for j in range(k + 1))
    return d_omega


def pullback(F, omega):
    def pb(x):
        J = jacrev(F)(x)
        w = omega(F(x))
        for _ in range(w.dim()):
            w = torch.tensordot(w, J, dims=([0], [0]))
        return w
    return pb


def interior(v, w):
    return torch.tensordot(v, w, dims=([0], [0]))


def gauss(a, b, n):
    x, w = np.polynomial.legendre.leggauss(n)
    return T(0.5 * (b - a) * x + 0.5 * (a + b)), T(0.5 * (b - a) * w)


def periodic(n, length=2 * math.pi):
    return torch.arange(n) * (length / n), torch.full((n,), length / n)


def integrate(component, rules):
    # integral of a function of u over a box, with one (nodes, weights) rule per variable
    grids = torch.meshgrid(*[r[0] for r in rules], indexing="ij")
    weights = torch.meshgrid(*[r[1] for r in rules], indexing="ij")
    U = torch.stack([g.reshape(-1) for g in grids], dim=1)
    W = torch.stack([w.reshape(-1) for w in weights], dim=1).prod(dim=1)
    return (vmap(component)(U) * W).sum()


def top(form, param):
    # the component of the pull-back of a k-form by a parametrization of a k-dimensional piece
    def comp(u):
        w = pullback(param, form)(u)
        return w[tuple(range(w.dim()))]
    return comp


e2 = [dx(0, 2), dx(1, 2)]
e3 = [dx(i, 3) for i in range(3)]
mu3 = wedge(wedge(e3[0], e3[1]), e3[2])

2.15.0+cu132


In [2]:
from torch.func import jvp, vjp, hessian


def rk4(field, a, T, steps=200):
    # the flow of the autonomous field at time T (T may be negative, or a tensor)
    h = T / steps
    x = a
    for _ in range(steps):
        k1 = field(x)
        k2 = field(x + 0.5 * h * k1)
        k3 = field(x + 0.5 * h * k2)
        k4 = field(x + h * k3)
        x = x + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
    return x


gen = torch.Generator().manual_seed(0)
Wa, ba = torch.randn(8, 2, generator=gen) / 2, torch.randn(8, generator=gen)
Wb = torch.randn(2, 8, generator=gen) / 2
X2 = lambda x: Wb @ torch.tanh(Wa @ x + ba)                  # a smooth field on the plane, with bounded derivatives
L_bound = (torch.linalg.matrix_norm(Wb, ord=2) * torch.linalg.matrix_norm(Wa, ord=2)).item()
flow = lambda t: (lambda a: rk4(X2, a, t))
print(f"Lipschitz bound of the field: L <= {L_bound:.4f}")

Lipschitz bound of the field: L <= 2.5934


## 1. Picard iterates and the exponential (Theorems 25.2 and 25.6)

For $\Phi' = A\Phi$, $\Phi(0) = I$, each Picard iterate is a polynomial in
$t$ with matrix coefficients; the iteration is carried out on the
coefficients.

In [3]:
A = torch.randn(4, 4, generator=gen)
coeffs = [torch.eye(4)]                                       # Phi_0 = I
for k in range(25):
    # Phi_{k+1}(t) = I + int_0^t A Phi_k(s) ds, on the coefficients of t^j
    coeffs = [torch.eye(4)] + [A @ c / (j + 1) for j, c in enumerate(coeffs)]
for j in range(6):
    assert torch.allclose(coeffs[j], torch.linalg.matrix_power(A, j) / math.factorial(j))
t = 0.8
series = sum(c * t ** j for j, c in enumerate(coeffs))
expA = torch.linalg.matrix_exp(t * A)
assert torch.allclose(series, expA, atol=1e-10)
print("the 25th Picard iterate is the partial sum of the exponential series; error", (series - expA).abs().max().item())

# group law, inverse, determinant, and the equation
s_ = -0.3
assert torch.allclose(torch.linalg.matrix_exp((s_ + t) * A), torch.linalg.matrix_exp(s_ * A) @ expA)
assert torch.allclose(torch.linalg.inv(expA), torch.linalg.matrix_exp(-t * A))
assert torch.allclose(torch.linalg.det(expA), torch.exp(t * torch.trace(A)))
dPhi = jacrev(lambda tt: torch.linalg.matrix_exp(tt * A))(T(t))
assert torch.allclose(dPhi, A @ expA)
x0 = torch.randn(4, generator=gen)
assert torch.allclose(rk4(lambda x: A @ x, x0, t), expA @ x0, atol=1e-9)
print(f"det e^(tA) = {torch.linalg.det(expA).item():.6f} = e^(t tr A) = {math.exp(t * torch.trace(A).item()):.6f}")

# Exercise 25.2
for M, expected in ((T([[0.0, 1], [0, 0]]), T([[1.0, t], [0, 1]])),
                    (T([[0.0, -1], [1, 0]]), T([[math.cos(t), -math.sin(t)], [math.sin(t), math.cos(t)]])),
                    (T([[1.0, 0], [0, -1]]), torch.diag(T([math.exp(t), math.exp(-t)])))):
    assert torch.allclose(torch.linalg.matrix_exp(t * M), expected)

the 25th Picard iterate is the partial sum of the exponential series; error 1.1102230246251565e-15


det e^(tA) = 1.466916 = e^(t tr A) = 1.466916


## 2. Rotations (Example 25.7) and a quadratic loss (Example 25.8)

In [4]:
Om = torch.randn(5, 5, generator=gen)
Om = Om - Om.T                                                # antisymmetric
Rm = torch.linalg.matrix_exp(0.7 * Om)
assert torch.allclose(Rm.T @ Rm, torch.eye(5), atol=1e-12) and torch.allclose(torch.linalg.det(Rm), T(1.0))

u = torch.randn(3, generator=gen)
u = u / u.norm()
N = T([[0.0, -u[2], u[1]], [u[2], 0.0, -u[0]], [-u[1], u[0], 0.0]])
xv = torch.randn(3, generator=gen)
assert torch.allclose(N @ xv, torch.linalg.cross(u, xv))
assert torch.allclose(N @ N, torch.outer(u, u) - torch.eye(3)) and torch.allclose(N @ N @ N, -N)
ang = 1.1
rodrigues = torch.eye(3) + math.sin(ang) * N + (1 - math.cos(ang)) * N @ N
assert torch.allclose(rodrigues, torch.linalg.matrix_exp(ang * N))
moved = rk4(lambda x: torch.linalg.cross(u, x), xv, ang)
assert torch.allclose(moved, rodrigues @ xv, atol=1e-9)
assert torch.allclose(rodrigues @ u, u) and torch.allclose(moved @ u, xv @ u) and torch.allclose(moved.norm(), xv.norm())
# the angle between the parts orthogonal to the axis is t
perp = lambda w: w - (w @ u) * u
cosang = perp(moved) @ perp(xv) / (perp(moved).norm() * perp(xv).norm())
assert torch.allclose(torch.acos(cosang), T(ang))
first = T([1.0, 0, 0])
N3 = T([[0.0, -1, 0], [1, 0, 0], [0, 0, 0]])
assert torch.allclose(torch.linalg.matrix_exp(ang * N3) @ first, T([math.cos(ang), math.sin(ang), 0.0]))
print("e^(tN) = I + sin t N + (1 - cos t) N^2 is the rotation of angle t about u, the flow of x' = u x x")

# a quadratic loss: gradient descent against the flow
lam = T([0.45, 2.6])
for eta, k in ((0.01, 200), (0.3, 10)):
    gd, fl = (1 - eta * lam) ** k, torch.exp(-eta * k * lam)
    print(f"eta = {eta}: factors of gradient descent {gd.tolist()}, of the flow {fl.tolist()}")
assert torch.allclose((1 - 0.01 * lam) ** 200, torch.exp(-2.0 * lam), atol=0.02)
assert abs(1 - 0.775 * lam[1]) > 1                              # eta lambda_2 = 2.015: the iteration diverges
assert abs(0.3 * lam[1] - 0.78) < 1e-12 and abs(0.775 * lam[1] - 2.015) < 1e-12

e^(tN) = I + sin t N + (1 - cos t) N^2 is the rotation of angle t about u, the flow of x' = u x x
eta = 0.01: factors of gradient descent [0.405744715956309, 0.005149812009819582], of the flow [0.4065696597405991, 0.0055165644207607716]
eta = 0.3: factors of gradient descent [0.23450984231065378, 2.6559922791423966e-07], of the flow [0.2592402606458915, 0.00040973497897978643]


## 3. The flow (Theorems 25.4 and 25.10)

Group law, the bound $e^{L|t|}$, and the differential of the flow as the
solution of the variational equation.

In [5]:
a, b = T([0.6, -0.4]), T([0.1, 0.9])
s_, t = 0.7, 1.3
assert torch.allclose(flow(s_ + t)(a), flow(s_)(flow(t)(a)), atol=1e-9)
assert torch.allclose(flow(-t)(flow(t)(a)), a, atol=1e-9)
for tt in (0.5, 1.0, 2.0, -1.5):
    assert (flow(tt)(a) - flow(tt)(b)).norm() <= math.exp(L_bound * abs(tt)) * (a - b).norm()
print("phi_(s+t) = phi_s o phi_t, phi_(-t) is the inverse, and |phi_t(a) - phi_t(b)| <= e^(L|t|) |a - b|")


def with_variation(state):
    # the field on pairs (x, Phi, logdet): x' = X(x), Phi' = DX(x) Phi, (log det)' = div X(x)
    x, Phi = state[:2], state[2:6].reshape(2, 2)
    Jx = jacrev(X2)(x)
    return torch.cat([X2(x), (Jx @ Phi).reshape(-1), torch.trace(Jx).reshape(1)])


start = torch.cat([a, torch.eye(2).reshape(-1), torch.zeros(1)])
end = rk4(with_variation, start, t)
Phi_var = end[2:6].reshape(2, 2)
Phi_ad = jacrev(flow(t))(a)
assert torch.allclose(Phi_var, Phi_ad, atol=1e-8)
print("D phi_t(a) by automatic differentiation of the solver = solution of the variational equation:")
print(Phi_ad)

# Liouville (Theorem 25.12)
assert torch.allclose(torch.linalg.det(Phi_ad), torch.exp(end[6]), atol=1e-8) and torch.linalg.det(Phi_ad) > 0
print(f"det D phi_t(a) = {torch.linalg.det(Phi_ad).item():.8f} = exp(int div X) = {torch.exp(end[6]).item():.8f}")

phi_(s+t) = phi_s o phi_t, phi_(-t) is the inverse, and |phi_t(a) - phi_t(b)| <= e^(L|t|) |a - b|


D phi_t(a) by automatic differentiation of the solver = solution of the variational equation:
tensor([[ 1.6153, -0.6929],
        [-0.6115,  1.3794]])
det D phi_t(a) = 1.80447880 = exp(int div X) = 1.80447880


## 4. Volumes (Theorem 25.12)

The area of the image of a disc, from its boundary (the polygon formula of
Chapter 23) and from $\int_B\det D\varphi_t$; its derivative in $t$ against
$\int_{\varphi_t(B)}\operatorname{div}X$.

In [6]:
centre, rad = T([0.3, 0.2]), 0.8
disc = lambda v: centre + rad * v[0] * torch.stack([torch.cos(v[1]), torch.sin(v[1])])
detflow = lambda tt: (lambda v: torch.linalg.det(jacrev(flow(tt))(disc(v))) * rad ** 2 * v[0])
rules = [gauss(0, 1, 24), periodic(48)]
area = lambda tt: integrate(detflow(tt), rules)

angles = torch.arange(2000) * (2 * math.pi / 2000)
boundary = vmap(lambda th: flow(t)(disc(torch.stack([torch.ones(()), th]))))(angles)
shifted = torch.roll(boundary, -1, dims=0)
polygon = 0.5 * (boundary[:, 0] * shifted[:, 1] - shifted[:, 0] * boundary[:, 1]).sum()
assert abs(polygon - area(t)) < 1e-4
print(f"area of phi_t(B): {area(t).item():.6f} (integral of det), {polygon.item():.6f} (boundary polygon);  area of B: {math.pi * rad ** 2:.6f}")

eps = 1e-4
derivative = (area(t + eps) - area(t - eps)) / (2 * eps)
div_side = integrate(lambda v: torch.trace(jacrev(X2)(flow(t)(disc(v)))) * detflow(t)(v), rules)
assert abs(derivative - div_side) < 1e-6
print(f"d/dt area = {derivative.item():+.8f},  integral of div X over phi_t(B) = {div_side.item():+.8f}")

# Exercise 25.6: the radial field
radial = lambda x: x - T([1.0, -2.0])
assert torch.allclose(torch.linalg.det(jacrev(lambda x: rk4(radial, x, 0.5))(a)), T(math.exp(2 * 0.5)), atol=1e-9)

area of phi_t(B): 4.268779 (integral of det), 4.268771 (boundary polygon);  area of B: 2.010619


d/dt area = +2.33357825,  integral of div X over phi_t(B) = +2.33357825


## 5. Transport of a density (Theorem 25.13)

$p_0$ is the standard Gaussian density of the plane and
$p_t(x) = p_0(\varphi_{-t}(x))\det D\varphi_{-t}(x)$. The formula along a
trajectory, the continuity equation at a point (all derivatives by
automatic differentiation through the solver), and the weak form with
samples.

In [7]:
p0 = lambda x: torch.exp(-0.5 * (x ** 2).sum()) / (2 * math.pi)


def p(tt, x):
    back = lambda y: rk4(X2, y, -tt, steps=100)
    return p0(back(x)) * torch.linalg.det(jacrev(back)(x))


# along a trajectory: ln p_t(phi_t(a)) = ln p_0(a) - int_0^t div X
assert torch.allclose(torch.log(p(T(t), flow(t)(a))), torch.log(p0(a)) - end[6], atol=1e-7)

# the continuity equation at a point
tt, xq = T(0.9), T([0.4, -0.7])
dp_dt = torch.func.grad(lambda s: p(s, xq))(tt)
div_pX = torch.trace(jacrev(lambda x: p(tt, x) * X2(x))(xq))
print(f"d_t p = {dp_dt.item():+.8f},  div(p X) = {div_pX.item():+.8f},  sum = {(dp_dt + div_pX).item():.1e}")
assert abs(dp_dt + div_pX) < 1e-7

# total mass, by quadrature on a large square
g1 = gauss(-7, 7, 60)
mass = integrate(lambda x: p(tt, x), [g1, g1])
assert abs(mass - 1) < 1e-6

# weak form: d/dt E f(phi_t(Z)) = E df(X)(phi_t(Z)), with the same samples on both sides
Z = torch.randn(4000, 2, generator=gen)
ftest = lambda x: torch.exp(-((x - T([0.5, 0.5])) ** 2).sum())
mean_at = lambda s: vmap(lambda z: ftest(rk4(X2, z, s, steps=60)))(Z).mean()
lhs = (mean_at(0.9 + 1e-4) - mean_at(0.9 - 1e-4)) / 2e-4
rhs = vmap(lambda z: (lambda y: torch.func.grad(ftest)(y) @ X2(y))(rk4(X2, z, 0.9, steps=60)))(Z).mean()
assert abs(lhs - rhs) < 1e-7
print(f"weak form with 4000 samples: d/dt mean of f = {lhs.item():+.8f},  mean of df(X) = {rhs.item():+.8f}")

# Exercise 25.5: X = -x on the line
pt1 = lambda s, x: torch.exp(s) * torch.exp(-0.5 * (torch.exp(s) * x) ** 2) / math.sqrt(2 * math.pi)
s0, x1 = T(0.6), T(0.35)
res = torch.func.grad(lambda s: pt1(s, x1))(s0) + torch.func.grad(lambda x: pt1(s0, x) * (-x))(x1)
assert abs(res) < 1e-12

# the identity behind the probability flow: Laplacian p = div(p grad ln p)
lap = torch.trace(hessian(lambda x: p(tt, x))(xq))
alt = torch.trace(jacrev(lambda x: p(tt, x) * torch.func.grad(lambda y: torch.log(p(tt, y)))(x))(xq))
assert abs(lap - alt) < 1e-8

d_t p = -0.04482220,  div(p X) = +0.04482220,  sum = 9.5e-13


weak form with 4000 samples: d/dt mean of f = -0.14218615,  mean of df(X) = -0.14218615


## 6. The Lie derivative and Cartan's formula (Theorem 25.15)

In $\mathbb R^3$, for a $1$-form and a $2$-form: the derivative of
$\varphi_t^*\omega$ at $t = 0$ by finite differences, the formula with
$D\omega$ and $DX$, and $d(i_X\omega) + i_Xd\omega$.

In [8]:
X3 = lambda q: torch.stack([torch.sin(q[1]) + 0.3 * q[2], torch.tanh(q[0] * q[2]), torch.cos(q[0]) - 0.5 * q[1]])
one_form = lambda q: torch.stack([q[1] * q[2] ** 2, torch.cos(q[0]) * q[2], q[0] * q[1] * q[2]])
two_form = lambda q: (q[0] ** 2 * q[2] * wedge(e3[1], e3[2]) + torch.sin(q[0] + q[1]) * wedge(e3[2], e3[0])
                      + q[2] ** 3 * q[1] * wedge(e3[0], e3[1]))
q0 = T([0.7, -1.3, 0.4])
phi3 = lambda s: (lambda q: rk4(X3, q, s, steps=20))
eps = 1e-4
for name, om in (("1-form", one_form), ("2-form", two_form)):
    fd = (pullback(phi3(eps), om)(q0) - pullback(phi3(-eps), om)(q0)) / (2 * eps)
    DX = jacrev(X3)(q0)
    Dom = jacrev(om)(q0)                                         # derivative index last
    k = om(q0).dim()
    formula = torch.tensordot(Dom, X3(q0), dims=([k], [0]))       # (D omega X)
    w = om(q0)
    for j in range(k):
        formula = formula + torch.tensordot(w, DX, dims=([j], [0])).movedim(-1, j)   # omega(.., DX v_j, ..)
    cartan = d(lambda q: interior(X3(q), om(q)))(q0) + interior(X3(q0), d(om)(q0))
    assert torch.allclose(fd, formula, atol=1e-7) and torch.allclose(formula, cartan, atol=1e-10)
    print(f"{name}: finite difference, (1) and (2) of Theorem 25.15 agree; max |L_X omega| = {cartan.abs().max().item():.4f}")

# L_X mu = div X mu, and L_X commutes with d
lie = lambda om: (lambda q: d(lambda r: interior(X3(r), om(r)))(q) + interior(X3(q), d(om)(q)))
assert torch.allclose(lie(lambda q: mu3)(q0)[0, 1, 2], torch.trace(jacrev(X3)(q0)))
assert torch.allclose(lie(d(one_form))(q0), d(lie(one_form))(q0), atol=1e-9)

# (3) of Theorem 25.15: d/dt phi_t^* omega = phi_t^*(L_X omega), at t = 0.3
for om in (one_form, two_form):
    fd = (pullback(phi3(0.3 + eps), om)(q0) - pullback(phi3(0.3 - eps), om)(q0)) / (2 * eps)
    assert torch.allclose(fd, pullback(phi3(0.3), lie(om))(q0), atol=1e-7)

# Exercise 25.4: X = (x, -y), omega = x dy
Xs = lambda q: torch.stack([q[0], -q[1]])
om_s = lambda q: q[0] * e2[1]
w0 = T([0.6, -1.1])
assert torch.allclose(d(lambda q: interior(Xs(q), om_s(q)))(w0) + interior(Xs(w0), d(om_s)(w0)), torch.zeros(2))
assert torch.allclose(rk4(Xs, w0, 0.5), w0 * T([math.exp(0.5), math.exp(-0.5)]), atol=1e-10)

1-form: finite difference, (1) and (2) of Theorem 25.15 agree; max |L_X omega| = 1.4259
2-form: finite difference, (1) and (2) of Theorem 25.15 agree; max |L_X omega| = 1.0215


### A moving domain (Corollary 25.16)

The mass $\int_{\varphi_t(D)}\rho\,d\lambda$ of a disc carried by the flow
changes at a rate equal to the flux of $\rho X$ through its boundary.

In [9]:
rho_ = lambda x: 1.0 + 0.5 * torch.sin(x[0]) * x[1] ** 2
mass_at = lambda tt: integrate(lambda v: rho_(flow(tt)(disc(v))) * detflow(tt)(v), rules)
rate = (mass_at(t + eps) - mass_at(t - eps)) / (2 * eps)


def flux_density(th):
    curve = lambda s: flow(t)(disc(torch.stack([torch.ones(()), s])))
    c, dc = jvp(curve, (th[0],), (torch.ones(()),))
    Xc = X2(c)
    return rho_(c) * (Xc[0] * dc[1] - Xc[1] * dc[0])             # i_X(rho dx^dy) on the tangent of the boundary


flux = integrate(flux_density, [periodic(256)])
assert abs(rate - flux) < 1e-6
print(f"d/dt of the mass in the moving disc = {rate.item():+.8f},  flux through its boundary = {flux.item():+.8f}")

d/dt of the mass in the moving disc = -2.73882131,  flux through its boundary = -2.73882132


## 7. Gradient flow and Euler's method (Propositions 25.17 and 25.18)

In [10]:
Hq = T([[2.0, 0.6], [0.6, 1.0]])
fobj = lambda x: 0.5 * x @ Hq @ x + torch.log(torch.cosh(x[0] - 0.5))        # strongly convex, bounded second derivative
gradf = lambda x: torch.func.grad(fobj)(x)
neg = lambda x: -gradf(x)
mu_ = torch.linalg.eigvalsh(Hq).min().item()                                 # D^2 f >= Hq
Lf = torch.linalg.eigvalsh(Hq).max().item() + 1.0
xstar = rk4(neg, T([1.0, 1.0]), 60.0, steps=4000)
assert gradf(xstar).norm() < 1e-10

x_start = T([2.0, -1.5])
times = [0.0, 0.5, 1.0, 2.0, 4.0]
values = [fobj(rk4(neg, x_start, tt)) for tt in times]
assert all(values[i + 1] < values[i] for i in range(len(values) - 1))
for tt in times[1:]:
    assert (rk4(neg, x_start, tt) - xstar).norm() <= math.exp(-mu_ * tt) * (x_start - xstar).norm() + 1e-9
print(f"f decreases along the flow; |x(t) - x*| <= e^(-mu t) |x(0) - x*| with mu = {mu_:.4f}")

# d/dt f(x(t)) = -|grad f|^2
xt = rk4(neg, x_start, 0.5)
dfdt = (fobj(rk4(neg, x_start, 0.5 + 1e-5)) - fobj(rk4(neg, x_start, 0.5 - 1e-5))) / 2e-5
assert abs(dfdt + gradf(xt) @ gradf(xt)) < 1e-6

# Euler = gradient descent: error of order eta, below the bound
Tend = 1.0
exact = rk4(neg, x_start, Tend, steps=2000)
M = max((jacrev(neg)(y) @ neg(y)).norm().item() for y in (rk4(neg, x_start, s, steps=400) for s in torch.linspace(0, Tend, 101)))
errs = []
for k in (10, 20, 40, 80):
    eta = Tend / k
    xk = x_start
    for _ in range(k):
        xk = xk - eta * gradf(xk)
    errs.append((xk - exact).norm().item())
    bound = M * eta / (2 * Lf) * (math.exp(Lf * Tend) - 1)
    assert errs[-1] <= bound
    print(f"eta = {eta:.4f}: error of gradient descent {errs[-1]:.6f},  bound {bound:.4f}")
assert all(1.8 < errs[i] / errs[i + 1] < 2.3 for i in range(3))

# Exercise 25.7
assert abs(abs(0.9 ** 10 - math.exp(-1)) - 0.0192) < 1e-4 and abs(0.05 * (math.e - 1) - 0.0859) < 1e-4

f decreases along the flow; |x(t) - x*| <= e^(-mu t) |x(0) - x*| with mu = 0.7190


eta = 0.1000: error of gradient descent 0.037957,  bound 3.4369
eta = 0.0500: error of gradient descent 0.019139,  bound 1.7185
eta = 0.0250: error of gradient descent 0.009594,  bound 0.8592
eta = 0.0125: error of gradient descent 0.004801,  bound 0.4296


## 8. The adjoint equation (Proposition 25.19)

A field $X(x, \theta) = W_2\tanh(W_1x + b)$ on $\mathbb R^3$ and the loss
$\ell(x(T)) = \tfrac12\|x(T) - y\|^2$. The differentials with respect to
the input and to the parameters are computed in two ways: by automatic
differentiation through the solver, and by solving the adjoint equation
backwards, with the state, and accumulating
$\int\alpha\circ D_\theta X\,dt$. The backward pass uses only
vector–Jacobian products.

In [11]:
n, m = 3, 6
theta0 = 0.5 * torch.randn(m * n + m + n * m, generator=gen)


def field(x, theta):
    W1 = theta[:m * n].reshape(m, n)
    b1 = theta[m * n:m * n + m]
    W2 = theta[m * n + m:].reshape(n, m)
    return W2 @ torch.tanh(W1 @ x + b1)


a0, target, Tfin = torch.randn(n, generator=gen), torch.randn(n, generator=gen), 1.5
loss = lambda xT: 0.5 * ((xT - target) ** 2).sum()
solve = lambda a_, th: rk4(lambda x: field(x, th), a_, Tfin, steps=400)

# reference: automatic differentiation through the solver
g_a = torch.func.grad(lambda a_: loss(solve(a_, theta0)))(a0)
g_theta = torch.func.grad(lambda th: loss(solve(a0, th)))(theta0)

# the adjoint method: the state (x, alpha, beta) is carried backwards from T to 0
xT = solve(a0, theta0)


def backward_field(state):
    x, alpha = state[:n], state[n:2 * n]
    _, pull = vjp(lambda xx, th: field(xx, th), x, theta0)
    a_DxX, a_DthX = pull(alpha)                                   # alpha o D_x X  and  alpha o D_theta X
    return torch.cat([field(x, theta0), -a_DxX, -a_DthX])


final = torch.cat([xT, torch.func.grad(loss)(xT), torch.zeros_like(theta0)])
initial = rk4(backward_field, final, -Tfin, steps=400)
x_back, alpha0, beta0 = initial[:n], initial[n:2 * n], initial[2 * n:]
assert torch.allclose(x_back, a0, atol=1e-8)
assert torch.allclose(alpha0, g_a, atol=1e-7) and torch.allclose(beta0, g_theta, atol=1e-7)
print("differential with respect to the input:      ", g_a.tolist())
print("the same from the adjoint equation, alpha(0):", alpha0.tolist())
print(f"differential with respect to the {len(theta0)} parameters: max difference {(beta0 - g_theta).abs().max().item():.1e}")

# alpha(t) o D phi_t(a) does not depend on t
half = rk4(backward_field, final, -Tfin / 2, steps=200)
alpha_half = half[n:2 * n]
Phi_half = jacrev(lambda a_: rk4(lambda x: field(x, theta0), a_, Tfin / 2, steps=200))(a0)
assert torch.allclose(alpha_half @ Phi_half, g_a, atol=1e-7)

# a layer x + eta X(x) pulls a covector back by alpha o (I + eta DX): an Euler step of the adjoint equation
eta = 1e-3
layer = lambda x: x + eta * field(x, theta0)
al = torch.randn(n, generator=gen)
_, pull = vjp(layer, a0)
assert torch.allclose(pull(al)[0], al + eta * al @ jacrev(lambda x: field(x, theta0))(a0))

# Exercise 25.8
th, c_, a_, T_ = 0.4, 1.7, 0.9, 2.0
assert abs(c_ * math.exp(th * T_) - torch.func.grad(lambda aa: c_ * math.exp(th * T_) * aa)(T(a_))) < 1e-12
quad = integrate(lambda s: c_ * torch.exp(th * (T_ - s[0])) * torch.exp(th * s[0]) * a_, [gauss(0, T_, 8)])
assert abs(quad - c_ * T_ * math.exp(th * T_) * a_) < 1e-12

differential with respect to the input:       [3.186707441020353, -1.2929946955653404, 2.5667394065511107]
the same from the adjoint equation, alpha(0): [3.1867074410203493, -1.2929946955654297, 2.5667394065511258]
differential with respect to the 42 parameters: max difference 1.6e-12


## Summary

* The Picard iterates of a linear equation are the partial sums of the
  exponential; $\det e^{tA} = e^{t\,\mathrm{tr}A}$; antisymmetric matrices
  give rotations, with Rodrigues' formula in dimension 3.
* The flow satisfies the group law, separates points at most like
  $e^{L|t|}$, and its differential solves the variational equation.
* $\det D\varphi_t = \exp\int\operatorname{div}X$; the area of a set
  carried by the flow changes at the rate $\int\operatorname{div}X$.
* A transported density satisfies the continuity equation, pointwise and
  in the weak form with samples.
* Cartan's formula agrees with the derivative of the pull-back; the mass
  of a moving domain changes by the flux through its boundary.
* Gradient descent is within $O(\eta)$ of the gradient flow on a fixed
  interval of time.
* The adjoint equation gives the same differentials as automatic
  differentiation through the solver.